# CodeGen — Group 45
## Step 5: Model pivot — re-baseline on `Qwen2.5-Coder-1.5B` (same harness, run top to bottom)

**Where this fits:** Steps 1–4 built our full pipeline around `codegen-350M-multi`:
the execution harness (Step 1), the training-pair data engine (Step 2), LoRA
fine-tuning (Steps 3/3b) and RAG + a compile-guided cascade (Step 4). That work took
the model from 1.3% to 10.3% execution accuracy — an 8× improvement, but from a very
low base. Mentor feedback: the 350M model itself is the bottleneck, and any model is
allowed.

**This step swaps ONLY the subject model.** The benchmark, the prompt format,
`trim_to_body` and the `evaluate_one` harness are identical to Step 1, so every number
we ever produced stays directly comparable. The 350M results become the "before"
column of our final table.

| Setup (same harness throughout) | Score |
|---|---|
| codegen-350M vanilla (Step 1) | 1.3% |
| codegen-350M + translation fine-tune (Step 3b) | 7.1% |
| codegen-350M + compile-guided cascade (Step 4) | 10.3% |
| **Qwen2.5-Coder-1.5B vanilla (this notebook, 2026-07-16)** | **37.8%** |

**To reproduce:** `Runtime → Run all` on a **T4 GPU** runtime
(`Runtime → Change runtime type → T4 GPU`). Sections 0–6 need no GPU; only Section 7
(the baseline itself) does.

## 0. Colab setup — Drive + Hugging Face token (run this first)

Everything we produce (benchmark file, model copy, eval results) lives in Drive at
`MyDrive/CodeGen_Group45`, so a crashed or recycled Colab session never loses work.

**One-time setup:** add a Colab secret (key icon in the left sidebar) named `HF_TOKEN`
containing a Hugging Face **read** token, and switch **Notebook access** ON for it.
Unauthenticated downloads from Colab are exactly what stalls / 403s (July 2026).

In [ ]:
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
DRIVE_ROOT = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/CodeGen_Group45"
    for sub in ("data", "models", "eval"):
        os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)

    # HF auth BEFORE anything talks to the Hub. Colab secret: HF_TOKEN, Notebook access ON.
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF token loaded from Colab secret")
    except Exception as e:
        print(f"WARNING: could not read the HF_TOKEN secret ({type(e).__name__}). "
              "Hub downloads may stall or 403 — add the secret and enable Notebook access.")
else:
    print("Not on Colab — skipping Drive; the benchmark loads from the repo's data/ folder.")

# Escape hatch only — leave False. With an upgraded hf_xet + auth, the Xet backend is the
# path that works from Colab; the non-Xet fallback was 403ing server-side (July 2026).
DISABLE_XET = False
if DISABLE_XET:
    os.environ["HF_HUB_DISABLE_XET"] = "1"

print("DRIVE_ROOT =", DRIVE_ROOT)

Mounted at /content/drive
HF token loaded from Colab secret
DRIVE_ROOT = /content/drive/MyDrive/CodeGen_Group45


## 1. Install the Rust toolchain
This gives us `rustc` (the Rust compiler). Takes ~1 minute.

In [ ]:
# Install Rust (non-interactive)
!curl https://sh.rustup.rs -sSf | sh -s -- -y -q

# Make rustc/cargo visible to this notebook
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

# Verify
!rustc --version
!cargo --version

warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.

  stable-x86_64-unknown-linux-gnu installed - rustc 1.97.0 (2d8144b78 2026-07-07)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source
the corresponding env file under $HOME/.cargo.

This is usually done by running one of the following (note the leading DOT):
. "$HOME/.cargo/env"            # For sh/bash/zsh/ash/dash/pdksh
source "$HOME/.cargo/env.fish"  # For fish
source "~/.cargo/env.nu"  # For nushell
source "$HOME/.cargo/env.tcsh"  # For tcsh
. "$HOME/.cargo/env.ps1"        # For pwsh
source "$HOME/.cargo/env.xsh"   # For xonsh
rustc 1.97.0 (2d8144

## 2. Install Python dependencies

Only `huggingface_hub` + its `hf_xet` download backend — and we **upgrade** them, because
Colab's preinstalled `hf_xet` is exactly what stalled our model downloads.

**Deliberately NOT installed: `datasets`.** `pip install -U datasets` drags a newer pyarrow
over Colab's preinstalled one and crashes the runtime (`IpcReadOptions size changed`).
This notebook never imports `datasets` at all — the benchmark is a plain JSONL (Section 3).

In [ ]:
# Upgrade the Hub client + Xet backend BEFORE anything imports huggingface_hub.
# Do NOT add `datasets` or `torch` here (see the markdown above).
!pip install -q -U huggingface_hub hf_xet
print("done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 12.6 MB/s eta 0:00:00
done


## 3. Load the MultiPL-E Rust problems
`humaneval-rs` = 156 classic coding problems, translated into Rust, **with unit tests**.
Each problem has:
- **prompt** — the function signature + a doc comment (ends with an open `{`)
- **tests** — a `fn main()` full of `assert_eq!` checks (starts with the closing `}`)

So a complete program is simply: **prompt + the model's body + tests**.

We keep the benchmark as a plain JSONL file (repo: `data/humaneval_rs.jsonl`, Drive:
`CodeGen_Group45/data/humaneval_rs.jsonl`) and read it with stdlib `json` — no `datasets`
library, no Hub download, nothing to flake. `ds` is a plain list of dicts.

In [ ]:
import json, os

def load_benchmark():
    candidates = []
    if DRIVE_ROOT:
        candidates.append(os.path.join(DRIVE_ROOT, "data", "humaneval_rs.jsonl"))
    candidates += ["data/humaneval_rs.jsonl", "../data/humaneval_rs.jsonl"]  # repo checkout
    for path in candidates:
        if os.path.exists(path):
            with open(path) as f:
                problems = [json.loads(line) for line in f if line.strip()]
            print(f"Loaded {len(problems)} problems from cache: {path}")
            return problems

    # Last resort (no Hub involved): hand-upload the repo's data/humaneval_rs.jsonl,
    # then stash it on Drive so this never happens again.
    if IN_COLAB:
        from google.colab import files
        print("Benchmark not found on Drive. Upload data/humaneval_rs.jsonl from the repo:")
        uploaded = files.upload()
        raw = next(iter(uploaded.values()))
        problems = [json.loads(line) for line in raw.decode().splitlines() if line.strip()]
        if DRIVE_ROOT:
            dest = os.path.join(DRIVE_ROOT, "data", "humaneval_rs.jsonl")
            with open(dest, "wb") as f:
                f.write(raw)
            print("Cached to Drive:", dest)
        return problems
    raise FileNotFoundError("humaneval_rs.jsonl not found — expected in the repo's data/ "
                            "folder or on Drive under CodeGen_Group45/data/.")

ds = load_benchmark()
assert len(ds) == 156, f"expected 156 problems, got {len(ds)}"
assert all(k in ds[0] for k in ("name", "prompt", "tests", "stop_tokens"))

# Look at one problem so the format is concrete
ex = ds[0]
print("\n===== PROMPT (given) =====\n", ex["prompt"])
print("===== TESTS (given) =====\n", ex["tests"])
print("===== stop tokens =====", ex["stop_tokens"])

Loaded 156 problems from cache: /content/drive/MyDrive/CodeGen_Group45/data/humaneval_rs.jsonl

===== PROMPT (given) =====
 /// Check if in given vector of numbers, are any two numbers closer to each other than
/// given threshold.
/// >>> has_close_elements(vec![1.0, 2.0, 3.0], 0.5)
/// false
/// >>> has_close_elements(vec![1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
/// true
fn has_close_elements(numbers: Vec<f64>, threshold: f64) -> bool {

===== TESTS (given) =====
 }

fn main() {
    let candidate = has_close_elements;
    assert_eq!(candidate(vec![1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3), true);
    assert_eq!(candidate(vec![1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05), false);
    assert_eq!(candidate(vec![1.0, 2.0, 5.9, 4.0, 5.0], 0.95), true);
    assert_eq!(candidate(vec![1.0, 2.0, 5.9, 4.0, 5.0], 0.8), false);
    assert_eq!(candidate(vec![1.0, 2.0, 3.0, 4.0, 5.0, 2.0], 0.1), true);
    assert_eq!(candidate(vec![1.1, 2.2, 3.1, 4.1, 5.1], 1.0), true);
    assert_eq!(candidate(vec![1.1, 2.2, 3.1, 

## 4. The harness function
This is the heart of Step 1. It glues the three parts into one `main.rs`, compiles it,
runs it, and returns one of: `pass`, `compile_error`, `run_fail`, `compile_timeout`, `run_timeout`.

In [ ]:
import subprocess, tempfile, os

def evaluate_one(prompt, completion, tests, compile_timeout=60, run_timeout=10):
    """Assemble prompt+completion+tests into a Rust program, compile and run it."""
    program = prompt + completion + tests
    with tempfile.TemporaryDirectory() as wd:
        src  = os.path.join(wd, "main.rs")
        binp = os.path.join(wd, "prog")
        with open(src, "w") as f:
            f.write(program)

        # 1) compile
        try:
            c = subprocess.run(["rustc", src, "-o", binp],
                               capture_output=True, text=True, timeout=compile_timeout)
        except subprocess.TimeoutExpired:
            return "compile_timeout"
        if c.returncode != 0:
            return "compile_error"          # didn't even build

        # 2) run against the tests
        try:
            r = subprocess.run([binp], capture_output=True, text=True, timeout=run_timeout)
        except subprocess.TimeoutExpired:
            return "run_timeout"             # probably an infinite loop
        return "pass" if r.returncode == 0 else "run_fail"

print("harness ready")

harness ready


## 5. We self-test the harness (most important step)

---


Before we trust the harness, we prove it gives the right verdict on code we already know is
correct / wrong / broken. If these three checks don't come out as we expect, the bug is in our
**harness**, not in any model.

In [ ]:
ex = ds[0]   # HumanEval_0: has_close_elements(numbers: Vec<f64>, threshold: f64) -> bool

# (a) a CORRECT body  -> should PASS
correct_body = """
    for i in 0..numbers.len() {
        for j in 0..numbers.len() {
            if i != j && (numbers[i] - numbers[j]).abs() < threshold {
                return true;
            }
        }
    }
    return false;
"""

# (b) a WRONG body (compiles, but fails the tests) -> should RUN_FAIL
wrong_body = "\n    return false;\n"

# (c) a BROKEN body (does not compile) -> should COMPILE_ERROR
broken_body = "\n    return this_is_not_defined;\n"

print("correct ->", evaluate_one(ex["prompt"], correct_body, ex["tests"]))
print("wrong   ->", evaluate_one(ex["prompt"], wrong_body,   ex["tests"]))
print("broken  ->", evaluate_one(ex["prompt"], broken_body,  ex["tests"]))

assert evaluate_one(ex["prompt"], correct_body, ex["tests"]) == "pass"
assert evaluate_one(ex["prompt"], wrong_body,   ex["tests"]) == "run_fail"
assert evaluate_one(ex["prompt"], broken_body,  ex["tests"]) == "compile_error"
print("\n Harness works correctly — it can tell good Rust from bad.")

correct -> pass
wrong   -> run_fail
broken  -> compile_error

 Harness works correctly — it can tell good Rust from bad.


## 6. We run the harness over ALL problems (end-to-end pipeline test)
Here we feed a **dummy** body (`todo!()`) to every problem. It compiles but panics at runtime,
so almost everything comes back `run_fail`. The point isn't the score — it's that we prove our
harness runs cleanly across all 156 problems and gives us aggregate counts.

In [ ]:
from collections import Counter

def run_benchmark(completion_fn, limit=None):
    """completion_fn(example) -> a Rust function body (string)."""
    statuses = []
    data = ds if limit is None else ds[:limit]
    for ex in data:
        body = completion_fn(ex)
        statuses.append(evaluate_one(ex["prompt"], body, ex["tests"]))
    counts = Counter(statuses)
    pass_rate = counts["pass"] / len(statuses)
    return pass_rate, counts

# Dummy "model": always returns todo!()  (compiles, panics at runtime)
dummy_rate, dummy_counts = run_benchmark(lambda ex: "\n    todo!()\n")
print("Dummy completion — execution accuracy:", round(100*dummy_rate, 1), "%")
print("Breakdown:", dict(dummy_counts))

Dummy completion — execution accuracy: 0.0 %
Breakdown: {'run_fail': 156}


## 7. Our REAL baseline — vanilla `Qwen/Qwen2.5-Coder-1.5B`

**Model pivot (July 2026):** our original subject model `codegen-350M-multi` scored 1.3%
vanilla and ~10% after the full pipeline; mentor feedback was that this is too low and any
model is allowed. New subject model: **`Qwen/Qwen2.5-Coder-1.5B`** — the strongest open
code model in the ~1B class (Apache-2.0, fp16 ≈ 3 GB fits a T4, 32K context). We use the
**base** variant, not Instruct: MultiPL-E is a *completion* benchmark (the prompt ends
mid-function), so the base model plugs into the same prompt → `trim_to_body` → harness
pipeline with zero format changes. The codegen-350M numbers stay in our report as the
"before" column.

Needs a **GPU runtime**. Model acquisition is **Drive-first** with two fallbacks:

1. **Drive** `CodeGen_Group45/models/qwen25coder-1p5b` — used on every run after the first.
2. **ModelScope** — Alibaba's model hub, Qwen's home turf: the exact same files with zero
   HF infrastructure involved. Primary hub while HF-from-Colab keeps stalling (July 2026).
3. **Hugging Face Hub** — last resort. A stalled Xet download **hangs forever instead of
   raising**, so we run it in a subprocess we can kill on a 15-minute timeout; partial
   downloads resume on retry.

Whichever hub wins, we save an fp16 copy to Drive so no future session needs a hub at all.> **Bookkeeping note:** the 2026-07-16 run recorded below downloaded the model from the
> HF Hub with an earlier version of the fetch cell (HF before ModelScope). The cell was
> upgraded afterwards to the stall-proof ladder described above, so its output is
> cleared. The weights are identical either way — the 37.8% result is unaffected, and
> reruns now load from Drive anyway.

In [ ]:
# Do NOT add `torch` (Colab's preinstalled torch already matches its CUDA stack)
# and do NOT add `datasets` (see Section 2).
!pip install -q -U transformers accelerate
print("done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 103.5 MB/s eta 0:00:00
done


In [ ]:
import os, shutil, subprocess, sys

MODEL_ID  = "Qwen/Qwen2.5-Coder-1.5B"
MARKER    = "_SAVED_OK"   # written only after a COMPLETE save to Drive
DRIVE_MODEL_DIR = os.path.join(DRIVE_ROOT, "models", "qwen25coder-1p5b") if DRIVE_ROOT else None
LOCAL_DIR = "/content/qwen25coder-1p5b"

def _modelscope_download():
    # Alibaba's hub — Qwen's home turf, same files, zero HF infrastructure.
    # Verified 2026-07-14: modelscope 1.38's entire dep closure is
    # requests/tqdm/urllib3/packaging/filelock/modelscope-hub — no datasets, no
    # pyarrow — so a plain install cannot trigger the Colab pyarrow crash (Section 2).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "modelscope"],
                   check=True)
    from modelscope import snapshot_download
    return snapshot_download(MODEL_ID)

def _hf_download():
    # HF-from-Colab stalls mid-download (Xet, July 2026), and a stall HANGS forever
    # instead of raising — so the download runs in a subprocess we can kill on timeout.
    # snapshot_download resumes partial downloads, so a killed attempt costs nothing.
    code = f"from huggingface_hub import snapshot_download; snapshot_download('{MODEL_ID}')"
    for attempt in (1, 2):
        try:
            subprocess.run([sys.executable, "-c", code], check=True, timeout=900)
            from huggingface_hub import snapshot_download
            return snapshot_download(MODEL_ID, local_files_only=True)  # already cached
        except subprocess.TimeoutExpired:
            print(f"HF Hub attempt {attempt}: no finish within 15 min (stalled) — killed")
        except subprocess.CalledProcessError:
            print(f"HF Hub attempt {attempt}: download process errored")
    raise RuntimeError(
        "All hubs failed (Drive empty, ModelScope failed, HF stalled/errored twice). "
        "Check the Colab proxy/network, or download the model on another machine and "
        "upload it to Drive under models/qwen25coder-1p5b with an empty _SAVED_OK file.")

def fetch_model_dir():
    """Return a local directory holding the model files. Order: Drive -> ModelScope -> HF Hub."""
    # (1) Drive copy. Copy to local disk first: reading 3 GB straight off the Drive
    #     FUSE mount is slow and occasionally errors out mid-load.
    if DRIVE_MODEL_DIR and os.path.exists(os.path.join(DRIVE_MODEL_DIR, MARKER)):
        if not os.path.exists(os.path.join(LOCAL_DIR, MARKER)):
            print("Model found on Drive — copying to local disk (one-time per session)...")
            shutil.copytree(DRIVE_MODEL_DIR, LOCAL_DIR, dirs_exist_ok=True)
        print("Using the Drive copy")
        return LOCAL_DIR

    # (2) ModelScope — primary hub while HF-from-Colab is broken.
    try:
        path = _modelscope_download()
        print("Downloaded from ModelScope")
        return path
    except Exception as e:
        print(f"ModelScope failed: {type(e).__name__}: {e}")

    # (3) HF Hub — last resort, stall-proofed.
    path = _hf_download()
    print("Downloaded from the Hugging Face Hub")
    return path

model_dir = fetch_model_dir()
print("model files at:", model_dir)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), "No GPU — Runtime -> Change runtime type -> T4 GPU, then rerun."

tok = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForCausalLM.from_pretrained(model_dir, dtype=torch.float16).to("cuda")  # T4 has no bf16
model.eval()
print("model loaded on", model.device)

# One-time: stash an fp16 copy on Drive so no future session ever needs a hub again.
if DRIVE_MODEL_DIR and not os.path.exists(os.path.join(DRIVE_MODEL_DIR, MARKER)):
    print("Saving fp16 copy to Drive (one-time, ~3 GB, takes a few minutes)...")
    model.save_pretrained(DRIVE_MODEL_DIR)
    tok.save_pretrained(DRIVE_MODEL_DIR)
    with open(os.path.join(DRIVE_MODEL_DIR, MARKER), "w") as f:
        f.write("ok\n")
    print("Saved to", DRIVE_MODEL_DIR)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

model loaded on cuda:0
Saving fp16 copy to Drive (one-time, ~3 GB, takes a few minutes)...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/CodeGen_Group45/models/qwen25coder-1p5b


In [ ]:
def trim_to_body(text):
    # Cut at the brace that closes the function, IGNORING braces inside strings/chars/comments.
    depth = 1
    i, n = 0, len(text)
    in_str = in_char = in_line = in_block = False
    while i < n:
        ch = text[i]
        nxt = text[i+1] if i+1 < n else ""
        if in_line:
            if ch == "\n": in_line = False
            i += 1; continue
        if in_block:
            if ch == "*" and nxt == "/": in_block = False; i += 2; continue
            i += 1; continue
        if in_str:
            if ch == "\\": i += 2; continue
            if ch == '"': in_str = False
            i += 1; continue
        if in_char:
            if ch == "\\": i += 2; continue
            if ch == "'": in_char = False
            i += 1; continue
        if ch == "/" and nxt == "/": in_line = True; i += 2; continue
        if ch == "/" and nxt == "*": in_block = True; i += 2; continue
        if ch == '"': in_str = True; i += 1; continue
        if ch == "'":
            if nxt == "\\" or (i+2 < n and text[i+2] == "'"): in_char = True
            i += 1; continue
        if ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0: return text[:i]
        i += 1
    return text


def qwen_completion(ex, max_new_tokens=512):
    inputs = tok(ex["prompt"], return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id,
                             stop_strings=["\n}"], tokenizer=tok)  # MultiPL-E's stop token — saves GPU time
    text = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return trim_to_body(text)

print("completion fn ready")

completion fn ready


### 7b. Smoke test — 5 problems first (house rule)
Before burning 30–60 min of GPU time we check the whole pipeline end-to-end on 5 problems.
Anything other than `5 × compile_error` means the wiring is fine — even `todo!()` managed
`run_fail` across the board in Section 6.

In [ ]:
import time

t0 = time.time()
smoke_statuses = []
for ex in ds[:5]:
    body = qwen_completion(ex)
    status = evaluate_one(ex["prompt"], body, ex["tests"])
    smoke_statuses.append(status)
    print(f"{ex['name'][:45]:45s} {status:14s} ({time.time()-t0:4.0f}s total)")

print("\nSmoke:", smoke_statuses)
assert smoke_statuses.count("compile_error") < 5, (
    "All 5 came back compile_error — inspect one generated body (print `body`) "
    "before spending GPU time on the full run.")
print("Smoke test OK — safe to run the full benchmark.")

HumanEval_0_has_close_elements                pass           (   4s total)
HumanEval_1_separate_paren_groups             run_fail       (  11s total)
HumanEval_2_truncate_number                   pass           (  13s total)
HumanEval_3_below_zero                        pass           (  16s total)
HumanEval_4_mean_absolute_deviation           compile_error  (  20s total)

Smoke: ['pass', 'run_fail', 'pass', 'pass', 'compile_error']
Smoke test OK — safe to run the full benchmark.


### 7c. Full run — resumable, streams to Drive
One JSON line per problem (`name`, `status`, generated `body`) is appended to
`CodeGen_Group45/eval/step1_qwen25coder_1p5b_vanilla_rs.jsonl` as we go. If Colab dies at
problem 90, rerunning this cell skips the 90 finished problems and only does the remaining
66. The stored bodies double as raw material for error analysis later.

In [ ]:
import json
from collections import Counter

RESULTS_PATH = (os.path.join(DRIVE_ROOT, "eval", "step1_qwen25coder_1p5b_vanilla_rs.jsonl")
                if DRIVE_ROOT else "step1_qwen25coder_1p5b_vanilla_rs.jsonl")

done = {}
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        for line in f:
            rec = json.loads(line)
            done[rec["name"]] = rec["status"]
print(f"Resuming: {len(done)}/{len(ds)} problems already on Drive")

with open(RESULTS_PATH, "a") as out:
    for ex in ds:
        if ex["name"] in done:
            continue
        body = qwen_completion(ex)
        status = evaluate_one(ex["prompt"], body, ex["tests"])
        out.write(json.dumps({"name": ex["name"], "status": status, "body": body}) + "\n")
        out.flush()
        done[ex["name"]] = status
        print(f"[{len(done):3d}/{len(ds)}] {ex['name'][:45]:45s} {status}")

counts = Counter(done.values())
rate = counts["pass"] / len(done)
print(f"\nVanilla {MODEL_ID} on humaneval-rs: {100*rate:.1f}%  ({counts['pass']}/{len(done)} pass)")
print("Breakdown:", dict(counts))

Resuming: 0/156 problems already on Drive
[  1/156] HumanEval_0_has_close_elements                pass
[  2/156] HumanEval_1_separate_paren_groups             run_fail
[  3/156] HumanEval_2_truncate_number                   pass
[  4/156] HumanEval_3_below_zero                        pass
[  5/156] HumanEval_4_mean_absolute_deviation           compile_error
[  6/156] HumanEval_5_intersperse                       pass
[  7/156] HumanEval_6_parse_nested_parens               run_fail
[  8/156] HumanEval_7_filter_by_substring               pass
[  9/156] HumanEval_8_sum_product                       pass
[ 10/156] HumanEval_9_rolling_max                       run_fail
[ 11/156] HumanEval_10_make_palindrome                  compile_error
[ 12/156] HumanEval_11_string_xor                       pass
[ 13/156] HumanEval_12_longest                          compile_error
[ 14/156] HumanEval_13_greatest_common_divisor          pass
[ 15/156] HumanEval_14_all_prefixes                     pass
[ 16

## What this step adds
- The pipeline's "judge" re-verified unchanged (Sections 4–6) — all Step 1–4 numbers
  remain directly comparable.
- **New headline baseline: vanilla `Qwen2.5-Coder-1.5B` = 37.8% (59/156)** —
  59 pass / 62 run_fail / 34 compile_error / 1 run_timeout. Per-problem statuses and
  the generated bodies are streamed to Drive
  (`eval/step1_qwen25coder_1p5b_vanilla_rs.jsonl`) for error analysis.
- Download-once infrastructure: the fp16 model copy, the benchmark file and all eval
  results live on Drive — no future session depends on a model hub being reachable.

**Next:** error analysis of the 97 failures, then port the Step 3b/4 interventions
(translation LoRA, RAG, compile-guided cascade) to the new model, plus a
Qwen2.5-Coder-7B-in-4bit "large LLM" comparison row.